# VietLawAssist — Huấn luyện LoRA Tầng 4 & Xuất GGUF Siêu Tốc trên Kaggle

Phiên bản tối ưu hóa đặc biệt cho **Kaggle Python 3.12 + GPU T4/P100**:
- **Cố định `transformers==4.46.3`, `peft==0.13.2`, `bitsandbytes>=0.44.1`**: Chạy xong Cell 1 trong **10 giây**, không bao giờ bị treo hay lỗi `StrictDataclassDefinitionError`.
- **Không bị tràn VRAM (`CUDA out of memory`) trên T4 x2**: Tối ưu hóa `batch_size=1`, `gradient_accumulation_steps=8`, `max_seq_length=1024` và khóa `_n_gpu=1`.
- **Xuất trực tiếp file `qwen2.5-3b-legal-vn-Q4_K_M.gguf`** ngay trên Kaggle kèm link tải về máy.

In [ ]:
# CELL 1: CÀI ĐẶT SIÊU TỐC (CHỈ 10 GIÂY - TƯƠNG THÍCH 100% QWEN2.5 TRÊN KAGGLE)
!pip install -q "transformers==4.46.3" "peft==0.13.2" "accelerate==1.1.1" "bitsandbytes>=0.44.1"
import torch
assert torch.cuda.is_available(), "❌ BẠN CHƯA BẬT GPU! Hãy vào menu Settings -> Accelerator -> Chọn GPU T4 x2 hoặc P100!"
print(f"✅ [CELL 1] Cài đặt xong! Đang chạy trên GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# CELL 2: TẢI MODEL 3B CŨ (`bqbbao6/Qwen2.5-3B-legal-vn`) & GẮN LORA TẦNG 4
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

BASE_MODEL_NAME = "bqbbao6/Qwen2.5-3B-legal-vn"
MAX_SEQ_LENGTH = 1024

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"[*] Đang tải model: {BASE_MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    quantization_config=bnb_config,
    device_map={"": 0},
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("✅ [CELL 2] Đã nạp xong Model 3B và gắn LoRA Tầng 4!")

In [ ]:
# CELL 3: TIỀN XỬ LÝ DỮ LIỆU & ĐỊNH DẠNG PROMPT THEO KIẾN TRÚC 4 TẦNG
import json, os, glob
from datasets import Dataset

dataset_candidates = glob.glob("/kaggle/input/**/sft_vietlaw_*.json", recursive=True)
if not dataset_candidates:
    dataset_candidates = glob.glob("/kaggle/input/**/*.json", recursive=True)

if dataset_candidates:
    dataset_path = dataset_candidates[0]
elif os.path.exists("data/processed/sft_vietlaw_500.json"):
    dataset_path = "data/processed/sft_vietlaw_500.json"
else:
    raise FileNotFoundError("Không tìm thấy file JSON trong /kaggle/input! Hãy kiểm tra lại mục Input bên phải.")

with open(dataset_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

INTENT_BAREM_SPECS = {
    "QPPL_STRUCT": "[INTENT: QPPL_STRUCT] Barem 3 bộ phận Quy phạm pháp luật (Giả định - Quy định - Chế tài), xác định phương thức thể hiện và cơ cấu Quan hệ pháp luật (Chủ thể - Nội dung - Khách thể).",
    "VPPL_4_ELEMENTS": "[INTENT: VPPL_4_ELEMENTS] Barem 4 yếu tố cấu thành Vi phạm pháp luật (1. Mặt khách quan - 2. Mặt chủ quan/Lỗi - 3. Khách thể - 4. Chủ thể) và Trách nhiệm bồi thường thiệt hại ngoài hợp đồng (BLDS 2015).",
    "TRUE_FALSE": "[INTENT: TRUE_FALSE] Barem Nhận định Đúng/Sai 3 bước (I. Kết luận dứt khoát Đúng/Sai - II. Giải thích lý luận giáo trình - III. Căn cứ pháp lý & Phản ví dụ thực tế).",
    "CIVIL_INHERIT": "[INTENT: CIVIL_INHERIT] Barem thuật toán chia thừa kế (Xác định di sản riêng/chung vợ chồng -> Xét di chúc & người thừa kế không phụ thuộc di chúc Điều 644 BLDS 2015 hưởng 2/3 suất -> Chia theo pháp luật Điều 651 & thế vị Điều 652).",
    "CRIMINAL_AGE": "[INTENT: CRIMINAL_AGE] Barem xác định năng lực trách nhiệm hình sự theo độ tuổi (Điều 12, Điều 9, Điều 101 BLHS 2015: mốc 14-16 tuổi và từ đủ 16 tuổi).",
    "GENERAL_THEORY": "[INTENT: GENERAL_THEORY] Barem Lý luận chung về Nhà nước và Pháp luật (Nguồn gốc, bản chất, chức năng, hình thức nhà nước, bộ máy nhà nước CHXHCN VN, hệ thống VBQPPL, thực hiện pháp luật và phòng chống tham nhũng).",
    "PRACTICAL_CONSULT": "[INTENT: PRACTICAL_CONSULT] Barem Tư vấn pháp lý thực tiễn (1. Xác định phạm vi và đối tượng điều chỉnh - 2. Quyền và nghĩa vụ của các bên - 3. Thẩm quyền cơ quan xử lý - 4. Hiệu lực văn bản áp dụng).",
    "EXPLAIN_LAW": "[INTENT: EXPLAIN_LAW] Barem Giải thích pháp luật đơn giản (1. Bản chất văn bản/quy định - 2. Ai cần quan tâm - 3. Nội dung điều khoản then chốt - 4. Tình trạng hiệu lực pháp lý)."
}

SYSTEM_PROMPT = "Bạn là Giảng viên / Chuyên gia Pháp lý VietLawAssist môn Pháp luật Đại cương. Hãy giải bài tập rõ ràng theo barem chuẩn của Bộ GD&ĐT."

processed_samples = []
for item in raw_data:
    intent_code = item.get("intent_code", "VPPL_4_ELEMENTS").strip()
    instruction = item.get("instruction", "").strip()
    input_text  = item.get("input", "").strip()
    output_text = item.get("output", "").strip()
    barem_spec  = INTENT_BAREM_SPECS.get(intent_code, INTENT_BAREM_SPECS["VPPL_4_ELEMENTS"])

    # Chế độ 1: Chuẩn Tầng 4 (Context Fusion + Yêu cầu giải đề thi khớp với llmService.ts)
    user_rag_prompt = f"{barem_spec}\nNhiệm vụ: {instruction}\n\nYêu cầu: Hãy phân tích và giải đề thi sau đây theo chuẩn khung chấm điểm:\n{input_text}"
    msg_tier4 = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_rag_prompt},
        {"role": "assistant", "content": output_text},
    ]
    processed_samples.append({"text": tokenizer.apply_chat_template(msg_tier4, tokenize=False, add_generation_prompt=False)})

    # Chế độ 2: Hỏi trực tiếp
    msg_direct = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{instruction}\n\n{input_text}"},
        {"role": "assistant", "content": output_text},
    ]
    processed_samples.append({"text": tokenizer.apply_chat_template(msg_direct, tokenize=False, add_generation_prompt=False)})

def tokenize_fn(batch):
    out = tokenizer(batch["text"], truncation=True, max_length=1024, padding=False)
    out["labels"] = out["input_ids"].copy()
    return out

dataset = Dataset.from_list(processed_samples).shuffle(seed=42)
tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
print(f"✅ [CELL 3] Đã tiền xử lý {len(raw_data)} bài tập từ {dataset_path} -> {len(tokenized_dataset)} mẫu huấn luyện!")

In [ ]:
# CELL 4: HUẤN LUYỆN MÔ HÌNH LORA TẦNG 4 (HỖ TRỢ CHẠY NỐI TIẾP 2 CHẶNG)
import gc, os, glob, zipfile, torch
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from IPython.display import FileLink, display

gc.collect()
torch.cuda.empty_cache()

def clip_len(batch):
    return {
        "input_ids": [ids[:1024] for ids in batch["input_ids"]],
        "attention_mask": [m[:1024] for m in batch["attention_mask"]],
        "labels": [l[:1024] for l in batch["labels"]],
    }
safe_dataset = tokenized_dataset.map(clip_len, batched=True)

model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.config.use_cache = False

# 1. TỰ ĐỘNG TÌM & KHÔI PHỤC CHECKPOINT NẾU CHẠY NỐI TIẾP CHẶNG 2
zip_ckpts = sorted(glob.glob("/kaggle/input/**/checkpoint-*.zip", recursive=True))
if zip_ckpts and not glob.glob("/kaggle/input/**/checkpoint-*/trainer_state.json", recursive=True):
    for z in zip_ckpts:
        print(f"[*] Đang giải nén checkpoint: {z}")
        with zipfile.ZipFile(z, "r") as zip_ref:
            zip_ref.extractall("/kaggle/working/resumed_checkpoint")

all_ckpts = sorted(glob.glob("/kaggle/input/**/checkpoint-*", recursive=True)) + sorted(glob.glob("/kaggle/working/**/checkpoint-*", recursive=True))
ckpt_dirs = [c for c in all_ckpts if os.path.isdir(c) and os.path.exists(os.path.join(c, "trainer_state.json"))]
resume_ckpt = ckpt_dirs[-1] if ckpt_dirs else None

# 2. XÁC ĐỊNH MỤC TIÊU SỐ EPOCHS:
# - Chặng 1: Chưa có checkpoint -> Học đúng 1 Epoch (~4.5 tiếng) an toàn tuyệt đối
# - Chặng 2: Đã có checkpoint Chặng 1 -> Mục tiêu nâng lên 2 Epochs để học tiếp nửa còn lại!
TARGET_EPOCHS = 3 if len(safe_dataset) < 2000 else (2 if (resume_ckpt or len(safe_dataset) < 8000) else 1)

trainer = Trainer(
    model=model,
    train_dataset=safe_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True),
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=2e-4,
        fp16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        logging_steps=20,
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="/kaggle/working/checkpoints",
        save_strategy="steps",
        save_steps=250,
        save_total_limit=2,
        report_to="none",
        ddp_find_unused_parameters=False,
    ),
)

trainer.args._n_gpu = 1

# Khắc phục triệt để lỗi PyTorch 2.6+ UnpicklingError khi nạp checkpoint
try:
    import numpy as np
    import numpy._core.multiarray
    torch.serialization.add_safe_globals([numpy._core.multiarray._reconstruct, np.ndarray, np.dtype])
except Exception:
    pass
_orig_torch_load = torch.load
def _safe_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _orig_torch_load(*args, **kwargs)
torch.load = _safe_torch_load

if resume_ckpt:
    print(f"🔥 [CHẶNG 2] Đã tìm thấy checkpoint cũ! Tiếp tục huấn luyện từ: {resume_ckpt}")
    print(f"[*] Mục tiêu tiến độ: Nâng cấp lên tổng cộng {TARGET_EPOCHS} Epochs.")
    trainer.train(resume_from_checkpoint=resume_ckpt)
else:
    print(f"🚀 [CHẶNG 1] Bắt đầu huấn luyện Chặng 1 (Mục tiêu: {TARGET_EPOCHS} Epoch, ~4.5 tiếng)...")
    trainer.train()

# 3. LƯU BỘ TRỌNG SỐ LORA & CHECKPOINT CHO CHẶNG TIẾP THEO
model.save_pretrained("/kaggle/working/vietlaw_lora_adapter")
tokenizer.save_pretrained("/kaggle/working/vietlaw_lora_adapter")
!zip -q -r /kaggle/working/vietlaw_lora_adapter.zip /kaggle/working/vietlaw_lora_adapter

saved_ckpts = sorted(glob.glob("/kaggle/working/checkpoints/checkpoint-*"))
if saved_ckpts:
    latest_ckpt = [c for c in saved_ckpts if os.path.isdir(c)][-1]
    ckpt_zip = f"/kaggle/working/{os.path.basename(latest_ckpt)}.zip"
    os.system(f"zip -q -r {ckpt_zip} {latest_ckpt}")
    print(f"💾 Đã lưu file checkpoint chặng: {os.path.basename(ckpt_zip)}")
    display(FileLink(os.path.basename(ckpt_zip)))

print("✅ [CELL 4] Đã hoàn thành giai đoạn huấn luyện và lưu bộ trọng số!")


In [ ]:
# CELL 5: KIỂM THỬ SUY LUẬN TẦNG 4 NGAY TRÊN KAGGLE
model.eval()
test_query = (
    "[INTENT: VPPL_4_ELEMENTS] Barem 4 yếu tố cấu thành Vi phạm pháp luật và Bồi thường thiệt hại.\n"
    "Yêu cầu: Hãy phân tích và giải đề thi sau đây theo chuẩn khung chấm điểm:\n"
    "Tình huống: Cô giáo H là giáo viên mầm non, thấy bé T khóc mãi không dừng nên đã dùng băng keo dài 15cm, rộng 4cm dán miệng bé T lại. Sau 2 phút bé T tím tái và tử vong tại bệnh viện. Hãy xác định và phân tích lỗi của cô giáo H?"
)
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_query},
]
inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
with torch.no_grad():
    outputs = model.generate(input_ids=inputs, max_new_tokens=512, temperature=0.15, do_sample=True)
print("=== KẾT QUẢ KIỂM THỬ MÔ HÌNH TẦNG 4 ===\n")
print(tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True))

In [ ]:
# CELL 6: GỘP LORA & XUẤT FILE qwen2.5-3b-legal-vn-Q4_K_M.gguf TRỰC TIẾP TRÊN KAGGLE
!pip uninstall -y -q torchao
import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False

import gc, os, glob, re, shutil, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from IPython.display import FileLink, display

# 1. Tìm checkpoint mới nhất (ưu tiên checkpoint có số bước lớn nhất, ví dụ checkpoint-4750)
all_ckpts = glob.glob("/kaggle/input/**/checkpoint-*", recursive=True) + glob.glob("/kaggle/working/**/checkpoint-*", recursive=True)
valid_ckpts = [c for c in all_ckpts if os.path.exists(os.path.join(c, "adapter_model.safetensors"))]

def get_step(path):
    m = re.search(r'checkpoint-(\d+)', path)
    return int(m.group(1)) if m else -1

if valid_ckpts:
    valid_ckpts.sort(key=get_step)
    adapter_path = valid_ckpts[-1]
elif os.path.exists("/kaggle/working/vietlaw_lora_adapter"):
    adapter_path = "/kaggle/working/vietlaw_lora_adapter"
else:
    raise FileNotFoundError("Không tìm thấy thư mục LoRA adapter hoặc checkpoint hợp lệ!")

print(f"[*] Đang sử dụng LoRA Adapter tại bước cao nhất: {adapter_path} (Step {get_step(adapter_path)})")

# 2. Nạp Base Model FP16 và gộp LoRA
del model, trainer
gc.collect()
torch.cuda.empty_cache()

CLEAN_BASE = "Qwen/Qwen2.5-3B-Instruct"
print("[1/3] Đang nạp Base Model và gộp LoRA...")
tokenizer = AutoTokenizer.from_pretrained(CLEAN_BASE, trust_remote_code=True)
base_fp16 = AutoModelForCausalLM.from_pretrained(CLEAN_BASE, torch_dtype=torch.float16, device_map="cpu")
merged_model = PeftModel.from_pretrained(base_fp16, adapter_path)
merged_model = merged_model.merge_and_unload()

merged_dir = "/kaggle/working/merged_hf"
merged_model.save_pretrained(merged_dir, safe_serialization=True, max_shard_size="2GB")
tokenizer.save_pretrained(merged_dir)
del merged_model, base_fp16
gc.collect()

# 3. Xuất ra định dạng GGUF bằng llama.cpp
print("[2/3] Đang chuyển đổi sang định dạng GGUF...")
!git clone --depth 1 -q https://github.com/ggerganov/llama.cpp.git /tmp/llama.cpp
!pip install -q gguf protobuf sentencepiece

final_gguf = "/kaggle/working/qwen2.5-3b-legal-vn-Q4_K_M.gguf"
!python /tmp/llama.cpp/convert_hf_to_gguf.py {merged_dir} --outfile {final_gguf} --outtype q8_0

# Dọn thư mục tạm
shutil.rmtree(merged_dir, ignore_errors=True)

print(f"\n🎉 [HOÀN TẤT 100%] Đã tạo thành công file: {final_gguf}")
print("👉 Bấm vào link màu xanh bên dưới để tải trực tiếp về máy:")
display(FileLink("qwen2.5-3b-legal-vn-Q4_K_M.gguf"))
